# Part 4 — Fine-Tuning

Fine-tune `Qwen/Qwen2.5-1.5B-Instruct` via LoRA to answer English queries in Hebrew.

**Run the scripts first (from the workspace root):**
```
python code/part4/create_data.py          # generate data/train_data.jsonl
python code/part4/train.py --overfit      # sanity check (optional)
python code/part4/train.py                # full training → lora_adapter/
python code/part4/run_eval.py             # eval_outputs.jsonl
```
This notebook loads the output files and displays results — it does **not** run training.

## 4.1 — Training Data

In [ ]:
import json
import pandas as pd

TRAIN_DATA = "../../data/train_data.jsonl"

train_records = []
with open(TRAIN_DATA, encoding="utf-8") as f:
    for line in f:
        train_records.append(json.loads(line))

df_train = pd.DataFrame(train_records)
print(f"Training examples: {len(df_train)}")
df_train.head()

In [ ]:
# Show 5 random examples (prompt + Hebrew response)
pd.set_option("display.max_colwidth", 200)
df_train.sample(5, random_state=42)[["prompt", "response"]]

## 4.2 — Evaluation Results

In [ ]:
EVAL_FILE = "../../eval_outputs.jsonl"

eval_records = []
with open(EVAL_FILE, encoding="utf-8") as f:
    for line in f:
        eval_records.append(json.loads(line))

df_eval = pd.DataFrame(eval_records)
print(f"Evaluation records: {len(df_eval)}")
df_eval[["prompt", "base_output", "finetuned_output"]]

In [ ]:
# Side-by-side comparison: print all 20 prompts
for i, row in df_eval.iterrows():
    print(f"\n{'='*70}")
    print(f"Query {i+1}: {row['prompt']}")
    print('='*70)
    print(f"\n  [Base]")
    print(f"    {row['base_output'][:300]}")
    print(f"\n  [Fine-tuned]")
    print(f"    {row['finetuned_output'][:300]}")

In [ ]:
# Hebrew ratio metric: fraction of alphabetic characters that are Hebrew (U+05D0–U+05EA)
import unicodedata

def hebrew_ratio(text: str) -> float:
    """Fraction of letter characters (Unicode category L*) that are Hebrew."""
    letters  = [c for c in text if unicodedata.category(c).startswith('L')]
    if not letters:
        return 0.0
    hebrew   = [c for c in letters if '\u05d0' <= c <= '\u05ea']
    return len(hebrew) / len(letters)

df_eval["base_he_ratio"]      = df_eval["base_output"].apply(hebrew_ratio)
df_eval["finetuned_he_ratio"] = df_eval["finetuned_output"].apply(hebrew_ratio)

summary = pd.DataFrame({
    "metric": ["Mean Hebrew ratio", "% outputs majority Hebrew (>50%)"],
    "Base model": [
        f"{df_eval['base_he_ratio'].mean():.1%}",
        f"{(df_eval['base_he_ratio'] > 0.5).mean():.0%}",
    ],
    "Fine-tuned": [
        f"{df_eval['finetuned_he_ratio'].mean():.1%}",
        f"{(df_eval['finetuned_he_ratio'] > 0.5).mean():.0%}",
    ],
})
summary